In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
print("API Keys loaded")

API Keys loaded


In [3]:
from langchain_core.documents import Document

doc = Document(
    page_content=(
        "Full-time employees receive 24 days of paid annual leave per calendar year."
        "Up to 5 unused days may be carried over into the next year."
    ),
    metadata = {"fileName": "employee_handbook.pdf", "page":1, "area":"policies"}
)

print(doc.page_content)
print(doc.metadata)

Full-time employees receive 24 days of paid annual leave per calendar year.Up to 5 unused days may be carried over into the next year.
{'fileName': 'employee_handbook.pdf', 'page': 1, 'area': 'policies'}


In [4]:
from langchain_community.document_loaders import TextLoader, UnstructuredMarkdownLoader

# TextLoader is the recommended choice for markdown - keeps heading intact
notes = TextLoader("knowledge/wiki/onboarding.md", encoding="utf-8").load()

print(len(notes))
print(notes[0].metadata)
print(notes[0].page_content[:200])

1
{'source': 'knowledge/wiki/onboarding.md'}
# Engineering Onboarding

Welcome to Zomato Engineering. This page covers your first weeks. Ask in #new-joiners if anything is unclear.

## Day one

- Collect your laptop and access card from the IT d


In [9]:
from langchain_community.document_loaders import PyPDFLoader

pages = PyPDFLoader("knowledge/policies/employee_handbook.pdf").load()

print(len(pages))
print(pages[0].metadata)
print(pages[0].page_content[:200])


8
{'producer': 'Skia/PDF m154', 'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/154.0.0.0 Safari/537.36', 'creationdate': '2026-10-02T18:18:26+00:00', 'title': 'handbook.html', 'moddate': '2026-10-02T18:18:26+00:00', 'source': 'knowledge/policies/employee_handbook.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}
Zomato Employee Handbook
Version 4.2 · Effective 1 April 2026 · Owner: People Operations
1. Welcome to Zomato
Zomato connects millions of customers with restaurants and delivery partners every day. We


In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

handbooks_docs = PyMuPDFLoader(
    "knowledge/policies/employee_handbook.pdf"
).load()

handbook_text = "\n\n".join(
    doc.page_content for doc in handbooks_docs
)

print(f"Total pages loaded: {len(handbooks_docs)}")
print(f"Total characters extracted: {len(handbook_text):,}")

print("\n--- First Page Content ---")
print(handbooks_docs[0].page_content)

print("\n--- First Page Metadata ---")
print(handbooks_docs[0].metadata)

print("\n--- Combined Handbook Text (First 500 Characters) ---")
print(handbook_text[:500])

In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 900,           # maximum characters per chunk
    chunk_overlap = 120,        # characters shared between neighbouring chunks
    add_start_index = True      # records where in the page each chunk began
)

chunks = splitter.split_documents(handbooks_docs)

print(f"split {len(handbooks_docs)} pages into {len(chunks)} chunks")
print("\nFirst chunk:")
print(chunks[1].page_content)
print("\nMetadata:", chunks[0].metadata)

split 8 pages into 16 chunks

First chunk:
Simple and honest. Say what you mean, share bad news early and prefer plain language over jargon.
Respect for everyone. Every person who works with Zomato, from a rider to a director, deserves courtesy and
dignity.
3. Employment basics
New employees serve a probation period of 6 months. Confirmation depends on a probation review with your manager
in month 5. During probation either side may end employment with 15 days' notice. Employment types are Full-time,
Fixed-term and Intern. Benefits in this handbook apply to full-time employees unless stated otherwise. Interns receive a
stipend and access to the learning portal but not leave or insurance benefits.

Metadata: {'producer': 'Skia/PDF m154', 'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/154.0.0.0 Safari/537.36', 'creationdate': '2026-10-02T18:18:26+00:00', 'source': 'knowledge/policies/employee_handbook.pdf', 'file_path': '

In [25]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

by_char = CharacterTextSplitter(separator="\n\n", chunk_size = 500, chunk_overlap = 50)
by_recursive = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap = 50)
by_tokens = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", chunk_size = 120, chunk_overlap = 15
)

for name, sp in [("character", by_char), ("recursive", by_recursive), ("tokens", by_tokens)]:
    pieces = sp.split_text(handbook_text)
    longest = max(len(p) for p in pieces)
    print(f"{name:>10}: {len(pieces):>3} chunks, longest = {longest} characters")

Created a chunk of size 1442, which is longer than the specified 500
Created a chunk of size 1630, which is longer than the specified 500
Created a chunk of size 1097, which is longer than the specified 500
Created a chunk of size 1556, which is longer than the specified 500
Created a chunk of size 1596, which is longer than the specified 500
Created a chunk of size 1283, which is longer than the specified 500
Created a chunk of size 1085, which is longer than the specified 500


 character:   8 chunks, longest = 1630 characters
 recursive:  29 chunks, longest = 494 characters
    tokens:  28 chunks, longest = 577 characters


In [27]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "h1"),
        ("##", "h2"),
        ("###", "h3")
    ]
)

markdown = open("knowledge/wiki/onboarding.md", encoding="utf-8").read()
sections = header_splitter.split_text(markdown)

print(len(sections), "sections")
for section in sections:
    print(section)
print(f"\n Section metadata example:",sections[2].metadata)
print(f"Content:", sections[2].page_content[:150])

17 sections
page_content='Welcome to Zomato Engineering. This page covers your first weeks. Ask in #new-joiners if anything is unclear.' metadata={'h1': 'Engineering Onboarding'}
page_content='- Collect your laptop and access card from the IT desk.
- Meet your onboarding buddy. They will walk you through the team's codebase and rituals.
- Complete the mandatory security awareness course in the Learning Portal within your first 3 days.
- Join the team Slack channels listed in your welcome email.' metadata={'h1': 'Engineering Onboarding', 'h2': 'Day one'}
page_content='You will receive these accounts on day one: company email, Slack, GitHub, Jira, Confluence and the internal Zomato Admin dashboard. Production access is **not** granted automatically. Request it through the Access Portal after you finish the security course; your manager must approve it, and access is reviewed every quarter.' metadata={'h1': 'Engineering Onboarding', 'h2': 'Accounts and access'}
page_content='Install the c

In [28]:
import hashlib
import uuid


def chunk_id(chunk) -> str:
    digest = hashlib.sha1(chunk.page_content.encode("utf-8")).hexdigest()
    return str(uuid.uuid5(uuid.NAMESPACE_URL, f"{digest}"))

#Tag every chunk with tenant
for chunk in chunks:
    chunk.metadata["tenant_id"] = "zomato"

ids = [chunk_id(c) for c in chunks]
print(f"{len(ids)} IDs, {len(set(ids))} unique - any duplicates ?")
print(len(ids), len(set(ids)))       # equal, unless two chunks in one file are identical

16 IDs, 16 unique - any duplicates ?
16 16


---
#Embedding


In [29]:
import os
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    openai_api_base = "https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

texts = [
    "Full-time employees receive 24 days of paid annual leaves.",
    "client meals are reimbursed up to the per-person limit with a receipt.",
    "Restart the payments service from the runbook, never from the console."
]

vectors = embeddings.embed_documents(texts)
print("No. of vectors:", len(vectors))
print("vector dimensitons:", len(vectors[0]))
print("First 5 values :", vectors[0][:5])

query_vector = embeddings.embed_query("How many vacation days do I get?")
print("Query vector dimension:", len(query_vector))

No. of vectors: 3
vector dimensitons: 1536
First 5 values : [-0.00833892822265625, 0.0408935546875, 0.05999755859375, 0.0294189453125, -0.0223846435546875]
Query vector dimension: 1536


In [ ]:
from langchain_qdrant import QdrantVectorStore

vector_store = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    ids=ids,
    location=":memory",
    collection_name = "helpdesk_kb",
)

print('vector store ready')
print("points store:-", vector_store.client.count("helpdesk_kb").count)

In [32]:
#create collection yourself and hand the client to langchain:

from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, models

client = QdrantClient(url="http://localhost:6333")
dimenstion = len(embeddings.embed_query("probe"))
if not client.collection_exists("helpdesk_kb"):
    client.create_collection(
        collection_name="helpdesk_kb",
        vectors_config=models.VectorParams(size=dimenstion, distance=models.Distance.COSINE)
    )

vector_store = QdrantVectorStore(client=client, collection_name="helpdesk_kb", embedding=embeddings)
print(vector_store.collection_name)

helpdesk_kb


In [35]:
#insert vector into the collection
import hashlib
import uuid

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_id(chunk)-> str:
    digest = hashlib.sha1(chunk.page_content.encode("utf-8")).hexdigest()
    return str(uuid.uuid5(uuid.NAMESPACE_URL, f"{chunk.metadata['filename']} | {digest}"))

#Load
handbook_pages = PyMuPDFLoader("knowledge/policies/employee_handbook.pdf").load()
for doc in handbook_pages:
    doc.metadata["filename"] = "employee_handbook.pdf"
    doc.metadata["area"] = "policies"
    doc.metadata["doc_type"] = "pdf"
    doc.metadata["tenant_id"] = "zomato"
print(f"Total pages: {len(handbook_pages)}")
print(handbook_pages[0])

#chunk
splitter = RecursiveCharacterTextSplitter(chunk_size = 900, chunk_overlap = 120, add_start_index=True)
handbook_chunks = splitter.split_documents(handbook_pages)
handbook_ids = [chunk_id(c) for c in handbook_chunks]

print(f"chunks:{len(handbook_chunks)}, unique IDs : {len(set(handbook_ids))}")

#upsert - uses the embedding and vector_store created above
vector_store.add_documents(handbook_chunks, ids=handbook_ids)
print(f"points in collection: {vector_store.client.count('helpdesk_kb').count}")

Total pages: 8
page_content='Zomato Employee Handbook
Version 4.2 · Effective 1 April 2026 · Owner: People Operations
1. Welcome to Zomato
Zomato connects millions of customers with restaurants and delivery partners every day. We are a technology company at
heart, and our work touches food, logistics, payments and support. This handbook explains how we work, what you can
expect from us and what we expect from you. If anything here conflicts with your employment contract, the contract
applies.
2. Our values
Customer obsession. Start with what the customer, restaurant partner or delivery partner needs, then work
backwards.
Ownership. If you see a problem, you own it until it is fixed or handed over clearly.
Speed with care. We move fast, and we do not cut corners on food safety, payments or privacy.
Simple and honest. Say what you mean, share bad news early and prefer plain language over jargon.
Respect for everyone. Every person who works with Zomato, from a rider to a director, deserve

In [37]:
#let's ask question now
question = "How many vacation days do I get?"

results = vector_store.similarity_search(question, k =3)

for rank, doc in enumerate(results, start=1):
    print(f"\n---Result{ rank}---")
    print(doc.metadata.get("filename"), "page:", doc.metadata.get("page"))
    print(doc.page_content[:250])


---Result1---
employee_handbook.pdf page: 2
Floating
holidays
2 days per year
For local or personal festivals. Apply 7 days ahead.
Study leave
Up to 10 days
For exams linked to an approved course, with manager
approval.
Sabbatical
Up to 3 months
unpaid
After 4 years of service, subject to busi

---Result2---
employee_handbook.pdf page: 1
Up to 5 unused days may be carried over into the next year. Anything above 5 days lapses on 31 March.
Leave of more than 3 consecutive days needs manager approval at least 14 days ahead. Shorter leave needs
approval at least 2 working days ahead, exc

---Result3---
employee_handbook.pdf page: 2
7. Parental leave
Type
Paid
leave
Notes
Maternity
26 weeks
Can start up to 8 weeks before the due date. A further 4 weeks of unpaid
leave can be requested.
Paternity
4 weeks
Take within 6 months of the birth. Can be split into two blocks.
Adoption
12


In [40]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

retrieved = retriever.invoke("What is the process for reimbursing a client dinner?")
print(f"Retrieved {len(retrieved)} documents")

for doc in retrieved:
    print(" ", doc.metadata.get("filename"), "page:", doc.page_content)

Retrieved 4 documents
  employee_handbook.pdf page: 11. Benefits
Health insurance: cover of ₹5 lakh for you, your spouse and up to two children, paid fully by the company.
Parents can be added at a discounted group premium paid by you.
Accident and life cover: accidental death cover of ₹50 lakh and term life cover of 3 times your annual fixed
pay.
Meal credits: ₹3,000 per month in Zomato credits, usable on the app. Unused credits expire at month end.
Learning budget: ₹40,000 per year for courses, books and conferences, with manager approval. Unspent
budget does not carry over.
Wellness: 6 free counselling sessions per year through the employee assistance programme, plus a monthly gym
or fitness reimbursement of up to ₹1,500.
Phone and internet: reimbursement of up to ₹1,000 per month for work-related use.
Relocation: for approved transfers, a relocation package covering travel, temporary housing for 30 days and
  employee_handbook.pdf page: An exit interview is held in your final week,

In [42]:
from qdrant_client import models

only_policies = models.Filter(
    must=[
        models.FieldCondition(
            key="metadata.area",                      # LangChain stores metadata under "metadata"
            match=models.MatchValue(value="policies"),
        )
    ]
)

results = vector_store.similarity_search(
    "Can I expense a client dinner?",
    k=4,
    filter=only_policies,
)

for doc in results:
    print(doc.metadata.get("filename"), "area:", doc.page_content)

employee_handbook.pdf area: 11. Benefits
Health insurance: cover of ₹5 lakh for you, your spouse and up to two children, paid fully by the company.
Parents can be added at a discounted group premium paid by you.
Accident and life cover: accidental death cover of ₹50 lakh and term life cover of 3 times your annual fixed
pay.
Meal credits: ₹3,000 per month in Zomato credits, usable on the app. Unused credits expire at month end.
Learning budget: ₹40,000 per year for courses, books and conferences, with manager approval. Unspent
budget does not carry over.
Wellness: 6 free counselling sessions per year through the employee assistance programme, plus a monthly gym
or fitness reimbursement of up to ₹1,500.
Phone and internet: reimbursement of up to ₹1,000 per month for work-related use.
Relocation: for approved transfers, a relocation package covering travel, temporary housing for 30 days and
employee_handbook.pdf area: allows.
14. Conflicts of interest and gifts
You must disclose any outsi

---
##call llm


In [45]:
from langchain_openrouter import ChatOpenRouter

llm = ChatOpenRouter(
    model="nvidia/nemotron-3.5-super-vl:free",     # any model id from openrouter.ai/models
    temperature=0,                   # 0 keeps answers focused and repeatable
)

response = llm.invoke([
    ("system", "You are a concise assistant."),
    ("human", "Explain RAG in one sentence."),
])

print(response.content)

RAG is an AI technique that enhances large language models by retrieving relevant information from external sources before generating a response, improving accuracy and reducing hallucinations.


In [47]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are Zomato's internal help-desk assistant.
Answer using only the context below.

Rules:
- If the answer is not in the context, say you don't know.
- Never invent policies, amounts or procedures.

Context:
{context}""",
    ),
    ("human", "{question}"),
])

print("prompt template created with placeholders: {context} and {question}")

prompt template created with placeholders: {context} and {question}


In [48]:
def format_docs(docs) ->str:
    return "\n\n---\n\n".join(
        f"[{i}] {d.metadata.get('filename', 'unknown')}\n{d.page_content}"
        for i, d in enumerate(docs, start=1)
    )

print("format_docs defined")

format_docs defined


In [56]:
question = "What is the policy on work from home?"

# Step 1 - Retrieve
retrieved_docs = retriever.invoke(question)
print(f"Step 1 - Retrieved: {len(retrieved_docs)} docs")

# Step 2 - Format
context = format_docs(retrieved_docs)
print(f"Step 2 - Context: {len(context)} characters")

# Step 3 - Build prompt
messages = prompt.invoke({
    "context": context,
    "question": question
})
print(f"Step 3 - Prompt messages: {len(messages.to_messages())}")

# Step 4 - Generate
response = llm.invoke(messages)
print(f"Step 4 - ANSWER: {response.content}")

Step 1 - Retrieved: 4 docs
Step 2 - Context: 3084 characters
Step 3 - Prompt messages: 2
Step 4 - ANSWER: Based on the employee handbook, here is the work-from-home (hybrid work) policy:

**Hybrid model**
- Most corporate teams follow a hybrid model: **3 days in the office (Tuesday to Thursday)** and **2 days remote**.
- Managers may adjust this for team needs and will inform you in writing.
- **Customer Support and Delivery Operations roles** are office or shift based and are **not covered** by the hybrid model.

**Remote work rules**
- Remote days still follow core hours, and you should be reachable on Slack.
- Working from **another city for more than 10 working days in a quarter** requires written approval from HR.
- Working from **another country** is not permitted without Legal and Finance approval (due to tax and visa rules).

**Setup allowance**
- A one-time work-from-home setup allowance of **₹15,000** is available for a desk and chair, claimable through the Expense Portal.


In [62]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {
        "context": retriever | format_docs,   # question -> Documents -> context string
        "question": RunnablePassthrough(),    # question -> question, unchanged
    }
    | prompt
    | llm
    | StrOutputParser()                       # AIMessage -> plain string
)

print(rag_chain.invoke("What is the policy on work from home?"))

Based on the employee handbook, here is the work-from-home (hybrid work) policy:

**Hybrid model**
- Most corporate teams work **3 days in the office (Tuesday to Thursday)** and **2 days remote**.
- Managers may adjust this schedule for team needs and will communicate any changes in writing.

**Roles not covered**
- Customer Support and Delivery Operations roles are office- or shift-based and are **not** covered by the hybrid model.

**Working from another location**
- Working from another city for **more than 10 working days in a quarter** requires written approval from HR.
- Working from another country is **not permitted without Legal and Finance approval**, due to tax and visa rules.

**Remote day expectations**
- Remote days still follow core hours, and you should be reachable on Slack.

**Setup allowance**
- A one-time work-from-home setup allowance of **₹15,000** is available for a desk and chair, claimable through the Expense Portal.


In [63]:
print("streaming answer..")

for token in rag_chain.stream("How many days of work from home can I have?"):
    print(token, end="", flush=True)
print()

streaming answer..
For most corporate teams, Zomato follows a hybrid model of **3 days in the office (Tuesday to Thursday) and 2 days remote** per week.

However, this may be adjusted by your manager for team needs, and they will inform you in writing. Customer Support and Delivery Operations roles are office or shift based and are not covered by the hybrid model.
